In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df

customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

customer_filtered = customer.loc[:, ["C_CUSTKEY"]]
orders_filtered = orders[
    ~orders["O_COMMENT"].str.contains(r"special[\S|\s]*requests")
]
orders_filtered = orders_filtered.loc[:, ["O_ORDERKEY", "O_CUSTKEY"]]
c_o_merged = customer_filtered.merge(
    orders_filtered, left_on="C_CUSTKEY", right_on="O_CUSTKEY", how="left"
)
c_o_merged = c_o_merged.loc[:, ["C_CUSTKEY", "O_ORDERKEY"]]
count_df = c_o_merged.groupby(["C_CUSTKEY"], as_index=False, sort=False).agg(
    C_COUNT=pd.NamedAgg(column="O_ORDERKEY", aggfunc="count")
)
total = count_df.groupby(["C_COUNT"], as_index=False, sort=False).size()
total.columns = ["C_COUNT", "CUSTDIST"]
total = total.sort_values(by=["CUSTDIST", "C_COUNT"], ascending=[False, False])